In [1]:
# Implementation of a KAN for Brain Voxel Classification (1-vs-Rest)
# This notebook implements a Kolmogorov-Arnold Network (KAN) for 
# brain voxel classification using a 1-vs-rest approach.

# Cell 1: Initialize the environment and import libraries
import torch
from kan import *
import numpy as np
import matplotlib.pyplot as plt
import os
import time
from sklearn.metrics import precision_recall_curve, average_precision_score, f1_score
from sklearn.preprocessing import StandardScaler
from datetime import datetime
import pandas as pd
import random
import glob
from tqdm import tqdm

# Check for GPU availability
if torch.cuda.is_available():
    device = torch.device("cuda")
    print(f"Using GPU: {torch.cuda.get_device_name(0)}")
else:
    device = torch.device("cpu")
    print("Using CPU")

# Create folders for saving results
os.makedirs('results', exist_ok=True)
os.makedirs('models', exist_ok=True)
os.makedirs('video_img', exist_ok=True)

# Set random seeds for reproducibility
torch.manual_seed(42)
np.random.seed(42)
random.seed(42)

Using GPU: NVIDIA RTX A6000


In [2]:
# Cell 2: Hyperparameters Configuration

class Config:
    def __init__(self):
        # Data parameters
        self.feature_dim = 341  # Number of input features
        self.num_classes = 2    # Binary classification (1-vs-rest)
        self.negative_ratio = 10  # Ratio of negative to positive samples (1:k)
        self.val_negative_ratio = 5  # Validation set negative to positive ratio
        self.test_size = 0.2    # Fraction of data to use for testing
        self.random_state = 42  # Random seed for data splitting
        self.normalize_features = True  # Whether to standardize features
        
        # KAN model parameters
        self.network_width = [self.feature_dim, 128, 64, self.num_classes]  # Network architecture
        self.grid_size = 20     # Fixed grid size (no grid expansion to avoid oscillation)
        self.k_value = 3        # Number of basis functions per dimension
        
        # Training parameters
        self.optimizer = "Adam"  # Optimizer type (Adam, SGD, etc.)
        self.learning_rate = 0.005  # Learning rate for optimizer
        self.weight_decay = 0.0001  # L2 regularization coefficient
        self.lambda_reg = 0.01    # Weight regularization coefficient
        self.lambda_entropy = 5.0  # Entropy regularization coefficient
        self.batch_size = 64      # Batch size for training
        self.train_steps = 100    # Number of training steps
        self.early_stopping = True  # Whether to use early stopping
        self.patience = 10        # Early stopping patience
        self.min_delta = 0.001    # Minimum improvement for early stopping

        # Class weighting for imbalanced data
        self.class_weights = torch.tensor([1.0, self.negative_ratio * 0.5], dtype=torch.float32)  # [pos_weight, neg_weight]
        
        # Pruning and fine-tuning
        self.enable_pruning = True  # Whether to prune the model after training
        self.fine_tune_steps = 50   # Number of fine-tuning steps after pruning
        
        # Evaluation metrics
        self.prediction_threshold = 0.5  # Threshold for binary predictions
        
        # Visualization parameters
        self.save_figures = True  # Whether to save figures
        self.img_folder = 'video_img'  # Folder to save training visualization images
        self.plot_frequency = 10   # How often to plot the model during training
        self.video_fps = 10        # Frames per second for training video
        
        # Symbolic regression parameters
        self.enable_symbolic = True  # Whether to extract symbolic expressions
        self.symbolic_library = ['x', 'x^2', 'exp', 'log', 'sqrt', 'sin', 'tanh', 'abs']  # Function library

    def display(self):
        """Display the current configuration"""
        print("=== KAN Brain Classification Configuration ===")
        print(f"Network Structure: {self.network_width}")
        print(f"Grid Size: {self.grid_size}, K Value: {self.k_value}")
        print(f"Negative to Positive Ratio: {self.negative_ratio}:1")
        print(f"Training Steps: {self.train_steps}, Batch Size: {self.batch_size}")
        print(f"Regularization: λ_reg={self.lambda_reg}, λ_entropy={self.lambda_entropy}")
        print(f"Learning Rate: {self.learning_rate}, Weight Decay: {self.weight_decay}")
        print(f"Early Stopping: {self.early_stopping} (patience={self.patience}, min_delta={self.min_delta})")
        print(f"Class Weights: {self.class_weights}")
        print("===============================================")

# Create a configuration object
config = Config()
config.display()

=== KAN Brain Classification Configuration ===
Network Structure: [341, 128, 64, 2]
Grid Size: 20, K Value: 3
Negative to Positive Ratio: 10:1
Training Steps: 100, Batch Size: 64
Regularization: λ_reg=0.01, λ_entropy=5.0
Learning Rate: 0.005, Weight Decay: 0.0001
Early Stopping: True (patience=10, min_delta=0.001)
Class Weights: tensor([1., 5.])


In [3]:
# Cell 3: Simplified Data Loading (Using existing validation set)

def load_brain_voxel_dataset(label_id, negative_ratio=10):
    """
    Load brain voxel dataset for binary classification using existing validation data as test set
    Following the style of Iris example's load_dataset() function
    """
    # 路径定义
    train_label_dir = '/home/jovyan/gpu_space/workspace_jiayi/KAN training/brain_voxel_data/output/train_set_by_label'
    val_label_dir = '/home/jovyan/gpu_space/workspace_jiayi/KAN training/brain_voxel_data/output/val_set_by_label'
    
    # 1. 加载训练数据 - 正样本
    positive_file = os.path.join(train_label_dir, f"label_{label_id}_count_*_voxels.npy")
    positive_files = glob.glob(positive_file)
    
    if not positive_files:
        raise ValueError(f"找不到标签{label_id}的训练数据文件")
    
    positive_samples = np.load(positive_files[0])
    num_positive = len(positive_samples)
    print(f"加载了 {num_positive} 个训练正样本")
    
    # 2. 加载训练数据 - 负样本
    negative_samples = []
    total_negative = num_positive * negative_ratio
    collected = 0
    
    label_files = glob.glob(os.path.join(train_label_dir, "label_*_count_*_voxels.npy"))
    other_label_files = [f for f in label_files if f"label_{label_id}_count" not in f]
    random.shuffle(other_label_files)
    
    for file in other_label_files:
        if collected >= total_negative:
            break
            
        samples = np.load(file)
        to_take = min(len(samples), total_negative - collected)
        
        if to_take < len(samples):
            indices = np.random.choice(len(samples), int(to_take), replace=False)
            samples = samples[indices]
            
        negative_samples.append(samples)
        collected += len(samples)
    
    negative_samples = np.vstack(negative_samples)
    print(f"收集了 {len(negative_samples)} 个训练负样本")
    
    # 3. 处理训练数据
    train_data = np.vstack([positive_samples, negative_samples])
    train_labels = np.concatenate([np.ones(len(positive_samples)), np.zeros(len(negative_samples))])
    
    # 随机打乱训练数据
    indices = np.arange(len(train_data))
    np.random.shuffle(indices)
    train_data = train_data[indices]
    train_labels = train_labels[indices].astype(np.int64)
    
    # 4. 加载验证数据 - 正样本
    val_positive_file = os.path.join(val_label_dir, f"label_{label_id}_count_*_voxels.npy")
    val_positive_files = glob.glob(val_positive_file)
    
    if not val_positive_files:
        print(f"警告: 找不到标签{label_id}的验证数据文件，将使用训练数据的一部分作为测试集")
        # 如果没有验证集，从训练集分出一部分
        split_point = int(0.8 * len(train_data))
        test_data = train_data[split_point:]
        test_labels = train_labels[split_point:]
        train_data = train_data[:split_point]
        train_labels = train_labels[:split_point]
    else:
        # 使用实际的验证数据
        val_positive_samples = np.load(val_positive_files[0])
        val_num_positive = len(val_positive_samples)
        print(f"加载了 {val_num_positive} 个验证正样本")
        
        # 5. 加载验证数据 - 负样本
        val_negative_samples = []
        val_total_negative = val_num_positive * negative_ratio
        val_collected = 0
        
        val_label_files = glob.glob(os.path.join(val_label_dir, "label_*_count_*_voxels.npy"))
        val_other_label_files = [f for f in val_label_files if f"label_{label_id}_count" not in f]
        random.shuffle(val_other_label_files)
        
        for file in val_other_label_files:
            if val_collected >= val_total_negative:
                break
                
            samples = np.load(file)
            to_take = min(len(samples), val_total_negative - val_collected)
            
            if to_take < len(samples):
                indices = np.random.choice(len(samples), int(to_take), replace=False)
                samples = samples[indices]
                
            val_negative_samples.append(samples)
            val_collected += len(samples)
        
        val_negative_samples = np.vstack(val_negative_samples)
        print(f"收集了 {len(val_negative_samples)} 个验证负样本")
        
        # 创建测试数据
        test_data = np.vstack([val_positive_samples, val_negative_samples])
        test_labels = np.concatenate([np.ones(len(val_positive_samples)), np.zeros(len(val_negative_samples))])
        
        # 随机打乱测试数据
        indices = np.arange(len(test_data))
        np.random.shuffle(indices)
        test_data = test_data[indices]
        test_labels = test_labels[indices].astype(np.int64)
    
    # 6. 转换为PyTorch张量
    train_inputs = torch.tensor(train_data, dtype=torch.float32).to(device)
    train_labels = torch.tensor(train_labels, dtype=torch.long).to(device)
    test_inputs = torch.tensor(test_data, dtype=torch.float32).to(device)
    test_labels = torch.tensor(test_labels, dtype=torch.long).to(device)
    
    # 创建类似Iris示例的数据集字典
    dataset = {
        'train_input': train_inputs,
        'train_label': train_labels,
        'test_input': test_inputs,
        'test_label': test_labels
    }
    
    print(f"训练集: {len(train_inputs)}个样本, 正:负比例 = 1:{negative_ratio}")
    print(f"测试集: {len(test_inputs)}个样本")
    
    return dataset

# 加载指定标签的数据集
target_label = 15  # 你想要分类的标签ID
dataset = load_brain_voxel_dataset(target_label, negative_ratio=10)

# 打印数据集信息
print(f"Train data shape: {dataset['train_input'].shape}")
print(f"Train target shape: {dataset['train_label'].shape}")
print(f"Test data shape: {dataset['test_input'].shape}")
print(f"Test target shape: {dataset['test_label'].shape}")
print("====================================")

加载了 9441 个训练正样本
收集了 94410 个训练负样本
加载了 272 个验证正样本
收集了 2720 个验证负样本
训练集: 103851个样本, 正:负比例 = 1:10
测试集: 2992个样本
Train data shape: torch.Size([103851, 341])
Train target shape: torch.Size([103851])
Test data shape: torch.Size([2992, 341])
Test target shape: torch.Size([2992])


In [4]:
# Cell 4 (Simplified): KAN Model Initialization

import torch
from kan import *
import matplotlib.pyplot as plt

# 使用与示例相同的简单初始化方式
model = KAN(width=[341, 128, 64, 2], grid=10, k=3, seed=0, device=device)

# 执行前向传播来初始化模型
_ = model(dataset['train_input'][:10])

print(f"KAN模型已创建: 输入维度={341}, 输出维度=2")
print(f"网格大小: 10, k值: 3")
print(f"总参数数量: {sum(p.numel() for p in model.parameters() if p.requires_grad)}")

# # 尝试绘制模型结构
# try:
#     model.plot(beta=100, scale=1, in_vars=['F1', 'F2', 'F3'], out_vars=['Neg', 'Pos'])
# except:
#     print("无法绘制模型结构图，但这不影响训练")

checkpoint directory created: ./model
saving model version 0.0
KAN模型已创建: 输入维度=341, 输出维度=2
网格大小: 10, k值: 3
总参数数量: 987392


In [5]:
# Cell 5: Enhanced Training with Comprehensive Metrics

from sklearn.metrics import precision_recall_curve, average_precision_score, f1_score, precision_score, recall_score

# 定义准确率度量函数 - 按照Iris示例
def train_acc():
    return torch.mean((torch.argmax(model(dataset['train_input']), dim=1) == dataset['train_label']).float())

def test_acc():
    return torch.mean((torch.argmax(model(dataset['test_input']), dim=1) == dataset['test_label']).float())

# 创建加权损失函数
pos_count = torch.sum(dataset['train_label'] == 1).item()
neg_count = torch.sum(dataset['train_label'] == 0).item()
print(f"训练集: 正样本={pos_count}个, 负样本={neg_count}个, 比例=1:{neg_count/pos_count:.1f}")

# 根据正负样本比例设置类别权重
class_weights = torch.tensor([1.0, neg_count/pos_count], device=device)
print(f"类别权重: {class_weights}")
loss_fn = torch.nn.CrossEntropyLoss(weight=class_weights)

# 与Iris示例类似的训练方法
print("开始训练模型...")
results = model.fit(dataset, opt="Adam", metrics=(train_acc, test_acc),
                    loss_fn=loss_fn, steps=100, 
                    lamb=0.01, lamb_entropy=10., save_fig=True, img_folder='video_img')

print("训练完成!")
print(f"最终训练准确率: {results['train_acc'][-1]:.4f}")
print(f"最终测试准确率: {results['test_acc'][-1]:.4f}")

# 计算详细的评估指标
def compute_detailed_metrics(model, data_input, data_label):
    """计算详细的性能指标"""
    model.eval()
    with torch.no_grad():
        # 获取模型预测
        logits = model(data_input)
        probs = torch.softmax(logits, dim=1)
        # 获取正类的概率 (第二列)
        pos_probs = probs[:, 1].cpu().numpy()
        # 获取预测类别
        predicted = torch.argmax(logits, dim=1).cpu().numpy()
        # 真实标签
        true_labels = data_label.cpu().numpy()
        
        # 计算基本指标
        accuracy = (predicted == true_labels).mean()
        precision = precision_score(true_labels, predicted, zero_division=0)
        recall = recall_score(true_labels, predicted, zero_division=0)
        f1 = f1_score(true_labels, predicted, zero_division=0)
        
        # 计算AUC-PR
        precision_curve, recall_curve, _ = precision_recall_curve(true_labels, pos_probs)
        auc_pr = average_precision_score(true_labels, pos_probs)
        
        # 计算Macro-F1和Weighted-F1
        macro_f1 = f1_score(true_labels, predicted, average='macro', zero_division=0)
        weighted_f1 = f1_score(true_labels, predicted, average='weighted', zero_division=0)
        
        return {
            'accuracy': accuracy,
            'precision': precision,
            'recall': recall,
            'f1': f1,
            'auc_pr': auc_pr,
            'macro_f1': macro_f1,
            'weighted_f1': weighted_f1
        }

# 计算训练集和测试集的详细指标
train_metrics = compute_detailed_metrics(model, dataset['train_input'], dataset['train_label'])
test_metrics = compute_detailed_metrics(model, dataset['test_input'], dataset['test_label'])

# 打印详细评估结果
print("\n详细性能评估:")
print("                     训练集         测试集")
print(f"准确率:          {train_metrics['accuracy']:.4f}       {test_metrics['accuracy']:.4f}")
print(f"F1分数(正样本):   {train_metrics['f1']:.4f}       {test_metrics['f1']:.4f}")
print(f"精确度(正样本):   {train_metrics['precision']:.4f}       {test_metrics['precision']:.4f}")
print(f"召回率(正样本):   {train_metrics['recall']:.4f}       {test_metrics['recall']:.4f}")
print(f"AUC-PR:          {train_metrics['auc_pr']:.4f}       {test_metrics['auc_pr']:.4f}")
print(f"Macro-F1:        {train_metrics['macro_f1']:.4f}       {test_metrics['macro_f1']:.4f}")
print(f"Weighted-F1:     {train_metrics['weighted_f1']:.4f}       {test_metrics['weighted_f1']:.4f}")

# 同样修改剪枝后的评估代码
def evaluate_after_pruning():
    # 这部分代码将在剪枝和微调后添加
    print("\n剪枝和微调后的详细性能评估:")
    train_metrics = compute_detailed_metrics(model, dataset['train_input'], dataset['train_label'])
    test_metrics = compute_detailed_metrics(model, dataset['test_input'], dataset['test_label'])
    
    print("                     训练集         测试集")
    print(f"准确率:          {train_metrics['accuracy']:.4f}       {test_metrics['accuracy']:.4f}")
    print(f"F1分数(正样本):   {train_metrics['f1']:.4f}       {test_metrics['f1']:.4f}")
    print(f"精确度(正样本):   {train_metrics['precision']:.4f}       {test_metrics['precision']:.4f}")
    print(f"召回率(正样本):   {train_metrics['recall']:.4f}       {test_metrics['recall']:.4f}")
    print(f"AUC-PR:          {train_metrics['auc_pr']:.4f}       {test_metrics['auc_pr']:.4f}")
    print(f"Macro-F1:        {train_metrics['macro_f1']:.4f}       {test_metrics['macro_f1']:.4f}")
    print(f"Weighted-F1:     {train_metrics['weighted_f1']:.4f}       {test_metrics['weighted_f1']:.4f}")

训练集: 正样本=9441个, 负样本=94410个, 比例=1:10.0
类别权重: tensor([ 1., 10.], device='cuda:0')
开始训练模型...


description:   0%|                                                          | 0/100 [00:00<?, ?it/s]


OutOfMemoryError: CUDA out of memory. Tried to allocate 16.89 GiB. GPU 0 has a total capacity of 47.53 GiB of which 16.38 GiB is free. Process 1856816 has 14.88 GiB memory in use. Process 1489785 has 2.66 GiB memory in use. Process 1189735 has 10.07 GiB memory in use. Process 2300719 has 636.00 MiB memory in use. Process 2473516 has 2.90 GiB memory in use. Of the allocated memory 2.45 GiB is allocated by PyTorch, and 142.69 MiB is reserved by PyTorch but unallocated. If reserved but unallocated memory is large try setting PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True to avoid fragmentation.  See documentation for Memory Management  (https://pytorch.org/docs/stable/notes/cuda.html#environment-variables)

In [ ]:
# Cell 6 (Simplified): Model Pruning and Symbolic Expression

# 模型剪枝
print("对模型进行剪枝...")
model = model.prune()
print("剪枝完成")

# 微调剪枝后的模型
print("微调剪枝后的模型...")
# 使用与之前相同的带权重损失函数
results_1 = model.fit(dataset, opt="Adam", metrics=(train_acc, test_acc),
                     loss_fn=loss_fn, steps=50, 
                     lamb=0.01, lamb_entropy=10.)

print(f"微调后训练准确率: {results_1['train_acc'][-1]:.4f}")
print(f"微调后测试准确率: {results_1['test_acc'][-1]:.4f}")

# 尝试绘制剪枝后的模型
try:
    model.plot(scale=1, in_vars=['F1', 'F2', 'F3'], out_vars=['Neg', 'Pos'])
except:
    print("无法绘制剪枝后的模型结构图")

# 提取符号表达式
print("尝试提取符号表达式...")
try:
    lib = ['x','x^2','x^3','exp','log','sqrt','tanh','sin','abs']
    model.auto_symbolic(lib=lib)
    
    # 获取符号公式
    formula1, formula2 = model.symbolic_formula()[0]
    
    print("\n负类符号表达式:")
    print(formula1)
    
    print("\n正类符号表达式:")
    print(formula2)
    
    # 尝试简化公式
    try:
        from sympy import simplify
        print("\n简化后的正类表达式:")
        print(simplify(formula2))
    except:
        print("无法简化公式")
except Exception as e:
    print(f"提取符号表达式失败: {e}")
# 评估剪枝和微调后的性能
evaluate_after_pruning()

In [ ]:
# Cell 7 (Simplified): Neural Network Comparison

# 定义一个标准神经网络进行比较
class BrainNet(nn.Module):
    def __init__(self):
        super(BrainNet, self).__init__()
        self.fc1 = nn.Linear(341, 256)  # 341输入到256隐藏节点
        self.relu = nn.ReLU()
        self.fc2 = nn.Linear(256, 128)  # 256到128隐藏节点
        self.fc3 = nn.Linear(128, 2)    # 128到2输出节点

    def forward(self, x):
        x = self.fc1(x)
        x = self.relu(x)
        x = self.fc2(x)
        x = self.relu(x)
        x = self.fc3(x)
        return x

# 训练神经网络模型
def train_model(model, train_loader, criterion, optimizer, num_epochs=100):
    model.train()
    for epoch in range(num_epochs):
        for inputs, labels in train_loader:
            inputs, labels = inputs.to(device), labels.to(device)
            optimizer.zero_grad()
            outputs = model(inputs)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()
        
        if (epoch+1) % 10 == 0:
            # 每10个epoch输出一次损失
            print(f'神经网络训练: Epoch {epoch+1}, Loss: {loss.item():.4f}')

# 评估神经网络模型
def test_model(model, test_loader):
    model.eval()
    correct = 0
    total = 0
    with torch.no_grad():
        for inputs, labels in test_loader:
            inputs, labels = inputs.to(device), labels.to(device)
            outputs = model(inputs)
            _, predicted = torch.max(outputs.data, 1)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()
    
    accuracy = 100 * correct / total
    print(f'神经网络测试准确率: {accuracy:.2f}%')
    return accuracy

# 创建数据加载器
train_loader = torch.utils.data.DataLoader(
    torch.utils.data.TensorDataset(dataset['train_input'], dataset['train_label']), 
    batch_size=32, 
    shuffle=True
)

test_loader = torch.utils.data.DataLoader(
    torch.utils.data.TensorDataset(dataset['test_input'], dataset['test_label']), 
    batch_size=32, 
    shuffle=False
)

# 初始化神经网络
nn_model = BrainNet().to(device)
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(nn_model.parameters(), lr=0.01)

# 训练神经网络
print("开始训练神经网络进行比较...")
train_model(nn_model, train_loader, criterion, optimizer)

# 评估神经网络
nn_accuracy = test_model(nn_model, test_loader)

# 比较KAN和神经网络
print("\n模型比较:")
print(f"KAN测试准确率: {results_1['test_acc'][-1]*100:.2f}%")
print(f"神经网络测试准确率: {nn_accuracy:.2f}%")
print(f"差异: {(results_1['test_acc'][-1]*100 - nn_accuracy):.2f}%")

# 打印KAN的优势
print("\nKAN的优势:")
print("1. 可解释性 - 能够提取数学公式解释决策")
print("2. 模型剪枝 - 可以减少模型大小并保持性能")
print("3. 固定网格大小避免训练震荡")

In [ ]:
# Cell 8 (Simplified): Create Video from Training Images

import os
import numpy as np

try:
    import moviepy.video.io.ImageSequenceClip
    
    # 创建视频
    video_name = 'video'
    fps = 10
    
    # 获取图像文件路径
    image_folder = 'video_img'
    files = os.listdir(image_folder)
    train_index = []
    
    # 获取所有数字命名的jpg文件
    for file in files:
        if file[0].isdigit() and file.endswith('.jpg'):
            train_index.append(int(file[:-4]))
    
    # 按正确顺序排序索引
    train_index = np.sort(train_index)
    
    # 创建图像文件路径列表
    image_files = [f'{image_folder}/{idx}.jpg' for idx in train_index]
    
    if image_files:
        # 创建视频并保存
        clip = moviepy.video.io.ImageSequenceClip.ImageSequenceClip(image_files, fps=fps)
        clip.write_videofile(f'{video_name}.mp4')
        
        print(f"已创建训练可视化视频: '{video_name}.mp4'")
        print(f"视频包含 {len(image_files)} 帧，帧率为 {fps} fps")
    else:
        print("未找到训练可视化图像文件")
        
except ImportError:
    print("未安装moviepy，跳过视频创建")
except Exception as e:
    print(f"创建训练视频时出错: {e}")